# D4·X. 드문 태그 보강 vs 무작위 보강 + TR 전문가 모델 (Colab A100 권장)

**질문 1 (D4)**: 학습 30만에 **드문 태그(NW·TR·FD) 실제 문장**을 더하면, 같은 수의 **무작위 문장**을 더했을 때보다 좋아지는가?
- D4_rare: 30만 + 드문 태그 문장 63,199 (NW 8,199 · TR 30,000 · FD 25,000) = 363,199문장
- D4_rand: 30만 + 무작위 문장 63,199 (60만 세트의 뒤쪽, 같은 표본 규칙) = 363,199문장
- 문장 수가 같으므로 차이는 '어떤 문장을 더했나'에서만 생긴다. 기준: D1_lc_300000.
- 거른 것: 한 글자 FD·TR·NW, TR '이름'((이름) 표시), FD 'lg화학'. 같은 개체 문자열은 NW 20·TR 10·FD 20문장까지만.

**질문 2 (X_TR)**: TR만 배우는 **전문가 모델**의 TR 답으로 기본 모델(CRF)의 TR 답을 바꾸면 좋아지는가?
- 학습: TR 문장 53,454(TR 라벨만 남김) + TR 없는 문장 53,454 = 106,908문장. 결합 방법은 dev로 고르고 test는 확인만.

**조건**: klue/roberta-base, 3에폭, 학습률 5e-5, 배치 16, 최대 길이 512, 시드 42, dev·test는 split_v1(=split_v3).
**예상 시간(추정)**: A100에서 X_TR 약 20분 + D4 두 개 각 약 1시간 = 약 2시간 20분. 끝나면 저장을 확인하고 런타임을 자동 반납한다.
**런타임**: A100, RAM 표준.


## 0. 공통 준비 (모든 노트북 같음)
Colab → 런타임 → 런타임 유형 변경 → GPU(T4 이상). 구글 드라이브를 연결하면 이 PC의 프로젝트 폴더가 `/content/drive/Othercomputers/내 PC/...` 아래에 보인다. 경로가 다르면 왼쪽 파일 탭에서 찾아 `PROJECT` 만 고친다.

공용 코드 `src/ner_lib.py`(학습)와 `src/evaluate.py`(팀 공용 채점기)를 불러온다. 정렬·복원·채점 규칙이 모든 노트북에서 같아야 비교가 공정하기 때문이다.

**저장되는 것**: 실험마다 `reports/results/{실험}.json`(에폭별 dev 점수 포함), `preds/`(예측), `scores/`(채점), `errors/`(오답), 그리고 `runs.jsonl`(실행 기록 한 줄). 그림은 R0 노트북이 이 파일들로 그린다.
**끊겼을 때**: 같은 셀을 다시 실행하면 끝난 실험은 건너뛰고, 진행 중이던 실험은 마지막 에폭 체크포인트부터 이어서 학습한다.

**최종 가중치(10/23부터 항상 저장)**: 학습이 끝나면 예측·채점보다 먼저 `내 드라이브/ner_models/{실험}/` 에 가중치(model.safetensors 또는 crf_model_state.pt)와 토크나이저를 저장하고, 다시 읽어 학습된 값과 같은지 확인한다. 확인에 실패하면 체크포인트를 지우지 않고 멈춘다. 런타임 반납 전에도 한 번 더 확인한다.

In [1]:
!pip -q install -U transformers datasets accelerate pytorch-crf kiwipiepy
import sys, os, json, torch
from google.colab import drive
drive.mount("/content/drive")
PROJECT = "/content/drive/Othercomputers/내 PC/NLP/프로젝트1/공통2_개체명인식_NER"
sys.path.insert(0, f"{PROJECT}/src")
import importlib, ner_lib; importlib.reload(ner_lib)
MODELS = "/content/drive/MyDrive/ner_models"   # 최종 가중치(실험마다 폴더). 학습 직후 저장하고 다시 읽어 확인한다. 지우지 말 것
ner_lib.DEFAULTS.update(model_dir=MODELS)
from ner_lib import run_experiment, load_results, results_table, upper_bound, load_jsonl, prepare_aihub_model
SPLIT   = f"{PROJECT}/data/split_v1"          # dev 1만 / test 1만 (모든 버전 공통)
TRAIN   = f"{PROJECT}/data/split_v3/train.jsonl"   # 팀 기준 학습 데이터: 30만 (2026-10-22 결정)
SUFFIX  = "_300k"                              # 30만으로 학습한 실험 이름 꼬리표
RESULTS = f"{PROJECT}/reports/results"        # 결과 json이 모이는 곳 (팀원 모두 같은 폴더)
CKPT    = "/content/drive/MyDrive/ner_ckpt"    # 에폭 체크포인트. 끊겨도 이어서 학습. '내 드라이브'라 PC로 동기화되지 않음
MEMBER  = "sejin_choi"                          # ← 실행한 사람 이름, 영어로 (결과 기록 runs.jsonl에 남음)
ner_lib.DEFAULTS.update(ckpt_dir=CKPT, member=MEMBER)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 30.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 140.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 139.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 49.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB


In [2]:
# 한글 글꼴 (그래프 글자가 네모로 깨지지 않게)
!apt-get -qq install -y fonts-nanum > /dev/null
import matplotlib.pyplot as plt, matplotlib.font_manager as fm, glob as _g
for f in _g.glob("/usr/share/fonts/truetype/nanum/*.ttf"): fm.fontManager.addfont(f)
plt.rc("font", family="NanumGothic"); plt.rc("axes", unicode_minus=False)
FIG = f"{PROJECT}/reports/figures"; os.makedirs(FIG, exist_ok=True)
def savefig(name):
    plt.tight_layout(); plt.savefig(f"{FIG}/{name}.png", dpi=150); plt.show(); print("저장:", f"{FIG}/{name}.png")

In [3]:
ner_lib.DEFAULTS["stage"] = "D4_드문태그보강"
D4 = f"{PROJECT}/data/d4_sets"
COMMON = dict(dev=f"{SPLIT}/dev.jsonl", test=f"{SPLIT}/test.jsonl", out_dir=RESULTS, model="klue/roberta-base",
              max_len=512, lr=5e-5, epochs=3, batch=16, seed=42)
RUNS = {   # 짧은 전문가 모델부터
    "X_TR_expert":  dict(train=f"{D4}/X_TR_train.jsonl", stage="X_태그전문가"),
    "D4_rare_363k": dict(train=f"{D4}/D4_rare_train.jsonl"),
    "D4_rand_363k": dict(train=f"{D4}/D4_rand_train.jsonl"),
}
print("GPU:", torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())


GPU: NVIDIA A100-SXM4-40GB | bf16: True


## 1. 시작 전 확인
세 학습 파일(합쳐 약 320MB)이 드라이브에 올라왔는지 본다. "없음"이면 PC의 구글 드라이브 동기화가 끝나지 않은 것이므로 잠시 뒤 다시 실행한다.


In [4]:
need = [COMMON["dev"], COMMON["test"]] + [v["train"] for v in RUNS.values()]
for f in need:
    ok = os.path.exists(f); print("있음" if ok else "없음!!", f.replace(PROJECT, "..."), f"{os.path.getsize(f)/1e6:.1f}MB" if ok else "")
assert all(os.path.exists(f) for f in need), "데이터 파일이 아직 드라이브에 없습니다"
for exp, v in RUNS.items():
    n = sum(1 for _ in open(v["train"], encoding="utf-8")); print(exp, f"{n:,}문장", "끝남(건너뜀)" if os.path.exists(f"{RESULTS}/{exp}.json") else "할 차례")


있음 .../data/split_v1/dev.jsonl 3.6MB
있음 .../data/split_v1/test.jsonl 3.8MB
있음 .../data/d4_sets/X_TR_train.jsonl 45.1MB
있음 .../data/d4_sets/D4_rare_train.jsonl 142.7MB
있음 .../data/d4_sets/D4_rand_train.jsonl 131.5MB
X_TR_expert 106,908문장 할 차례
D4_rare_363k 363,199문장 할 차례
D4_rand_363k 363,199문장 할 차례


## 2. 실행
끊기면 0번 셀부터 다시 실행한다. 끝난 실험은 건너뛰고, 진행 중이던 실험은 '내 드라이브/ner_ckpt' 의 마지막 에폭부터 이어서 학습한다.


In [5]:
for exp, v in RUNS.items():
    run_experiment({**COMMON, **v, "exp": exp})


config.json:   0%|          | 0.00/546 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/752k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

...

  [epoch 2] dev F1 0.7384 macro 0.555


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  [epoch 3] dev F1 0.7472 macro 0.5798


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[D4_rand_363k] dev F1 0.7472 | test F1 0.7378 | 학습 3673.2s (296.69 문장/s)


## 3. 결과 1 — 드문 태그 보강 vs 무작위 보강
판단 기준(미리 정함): D4_rare − D4_rand 의 dev F1 차이가 **0.7점 이상**이면 '드문 태그 보강이 효과 있음'. macro F1과 TR·FD·NW 태그 F1도 함께 본다(NW는 dev 13개라 크게 흔들림).


In [6]:
import pandas as pd
rows = []
for exp in ["D1_lc_300000", "D4_rand_363k", "D4_rare_363k"]:
    r = json.load(open(f"{RESULTS}/{exp}.json", encoding="utf-8"))
    d = r["dev"]; t = r["test"]
    rows.append({"exp": exp, "n_train": r["n_train"], "dev_F1": d["ALL"]["f1"], "dev_macro": d["MACRO_F1"],
                 **{f"dev_{k}": d[k]["f1"] for k in ("TM", "AF", "FD", "TR", "NW")},
                 "test_F1": t["ALL"]["f1"], "test_macro": t["MACRO_F1"], "train_min": round(r["train_sec"] / 60, 1)})
df = pd.DataFrame(rows).set_index("exp"); display(df.round(4))
diff = (df.loc["D4_rare_363k", "dev_F1"] - df.loc["D4_rand_363k", "dev_F1"]) * 100
print(f"드문 태그 보강 − 무작위 보강: dev F1 {diff:+.2f}점, dev macro {(df.loc['D4_rare_363k','dev_macro']-df.loc['D4_rand_363k','dev_macro'])*100:+.2f}점")


,n_train,dev_F1,dev_macro,dev_TM,dev_AF,dev_FD,dev_TR,dev_NW,test_F1,test_macro,train_min
exp,,,,,,,,,,,
D1_lc_300000,300000,0.7466,0.5641,0.5671,0.5107,0.5211,0.4702,0.3077,0.7306,0.5878,46.8
D4_rand_363k,363199,0.7472,0.5798,0.5635,0.5131,0.5000,0.4727,0.4348,0.7378,0.6006,61.2
D4_rare_363k,363199,0.7448,0.5692,0.5616,0.5313,0.4583,0.4606,0.3929,0.7315,0.5759,61.4


드문 태그 보강 − 무작위 보강: dev F1 -0.24점, dev macro -1.06점


## 4. 결과 2 — TR 전문가 결합
기본 모델은 CRF 30만이 있으면 그것, 없으면 CRF 20만. 결합 방법 3가지를 dev로 비교해 하나를 고르고, 고른 방법의 test만 본다.


In [7]:
main = "M2_roberta-base_crf_300k" if os.path.exists(f"{RESULTS}/preds/M2_roberta-base_crf_300k_dev.jsonl") else "M2_roberta-base_crf_200k"
print("기본 모델:", main)
!cd "{PROJECT}" && python src/combine_expert.py --main {main} --expert X_TR_expert --tag TR --results "{RESULTS}" --split-dir "{SPLIT}"


기본 모델: M2_roberta-base_crf_300k
{
 "main": "M2_roberta-base_crf_300k",
 "expert": "X_TR_expert",
 "tag": "TR",
 "main_only": {
  "dev": {
   "f1": 0.7597,
   "TR": 0.4944
  },
  "test": {
   "f1": 0.7474,
   "TR": 0.4836
  }
 },
 "replace_main_wins": {
  "dev": {
   "f1": 0.7499,
   "TR": 0.2828
  },
  "test": {
   "f1": 0.7398,
   "TR": 0.2724
  }
 },
 "replace_expert_wins": {
  "dev": {
   "f1": 0.7444,
   "TR": 0.2689
  },
  "test": {
   "f1": 0.7342,
   "TR": 0.2503
  }
 },
 "union": {
  "dev": {
   "f1": 0.7501,
   "TR": 0.2943
  },
  "test": {
   "f1": 0.7397,
   "TR": 0.2812
  }
 },
 "chosen_on_dev": "union"
}


## 5. 저장 확인 후 런타임 반납
결과 파일이 모두 드라이브에 있으면 저장을 마무리하고 런타임을 반납한다(컴퓨팅 단위 절약). 하나라도 없으면 멈춘다.


In [8]:
ner_lib.verify_saved_models()   # 이번에 다룬 실험의 최종 가중치가 하나라도 없으면 여기서 멈추고 반납하지 않음
print("가중치 위치:", MODELS)
missing = []
for exp in RUNS:
    for f in (f"{RESULTS}/{exp}.json", f"{RESULTS}/scores/{exp}_dev.json", f"{RESULTS}/scores/{exp}_test.json",
              f"{RESULTS}/preds/{exp}_dev.jsonl", f"{RESULTS}/preds/{exp}_test.jsonl",
              f"{RESULTS}/errors/{exp}_dev.jsonl", f"{RESULTS}/errors/{exp}_test.jsonl"):
        if not os.path.exists(f): missing.append(f.replace(RESULTS, "results"))
if missing:
    print("저장 안 된 파일이 있어 반납하지 않습니다:", missing[:10])
else:
    print("결과 파일 모두 저장됨. 드라이브 저장을 마무리하고 런타임을 반납합니다.")
    drive.flush_and_unmount()
    from google.colab import runtime
    runtime.unassign()

결과 파일 모두 저장됨. 드라이브 저장을 마무리하고 런타임을 반납합니다.
